In [2]:
import pandas as pd

In [6]:
import pandas as pd

# THE FIX: Add the skiprows parameter.
# You may need to open the CSV to find the exact number of rows to skip.
try:
    df_property_type = pd.read_csv(
        '/Users/callumhall/Documents/Dissertation/Datasets/Census Data/2021/Property Type.csv',
        skiprows=7  #<-- Adjust this number if needed
    )
    
    print("File loaded successfully.")
    print(df_property_type.head())

except Exception as e:
    print(f"An error occurred: {e}")
    print("\nTry opening the CSV file in a text editor to count the exact number of header lines to skip and use that number for 'skiprows'.")

File loaded successfully.
  2021 super output area - lower layer Total  Detached  Semi-detached  \
0              E01000248 : Barnet 001A   716      42.0          161.0   
1              E01000249 : Barnet 001B   621     179.0          244.0   
2              E01000250 : Barnet 001C  1145      86.0          296.0   
3              E01000254 : Barnet 001D   632     162.0          133.0   
4              E01000292 : Barnet 001E   655     102.0          237.0   

   Terraced  In a purpose-built block of flats or tenement  \
0     270.0                                          123.0   
1      90.0                                           84.0   
2     103.0                                          475.0   
3     259.0                                           42.0   
4      65.0                                          224.0   

   Part of a converted or shared house, including bedsits  \
0                                               87.0        
1                                       

In [8]:
df_property_type.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4993 entries, 0 to 4992
Data columns (total 10 columns):
 #   Column                                                                               Non-Null Count  Dtype  
---  ------                                                                               --------------  -----  
 0   2021 super output area - lower layer                                                 4988 non-null   object 
 1   Total                                                                                4992 non-null   object 
 2   Detached                                                                             4988 non-null   float64
 3   Semi-detached                                                                        4988 non-null   float64
 4   Terraced                                                                             4988 non-null   float64
 5   In a purpose-built block of flats or tenement                                        4988 

In [10]:
import pandas as pd
import geopandas as gpd

# --- 1. Load and Clean the Property Type Data ---
property_type_path = '/Users/callumhall/Documents/Dissertation/Datasets/Census Data/2021/Property Type.csv'

# Define the columns to keep and their new, clean names
cols_to_use = {
    '2021 super output area - lower layer': 'LSOA_Code_Name',
    'Total': 'Total_Households',
    'In a purpose-built block of flats or tenement': 'Flats_Purpose_Built',
    'Part of a converted or shared house, including bedsits': 'Flats_Converted_Shared',
    'Part of another converted building, for example, former school, church or warehouse': 'Flats_Converted_Other',
    'In a commercial building, for example, in an office building, hotel or over a shop': 'Flats_Commercial'
}

# Load the data, skipping the header rows
df_property = pd.read_csv(property_type_path, skiprows=7, usecols=list(cols_to_use.keys()))
df_property = df_property.rename(columns=cols_to_use)

# Clean the LSOA code
df_property[['LSOA_Code', 'LSOA_Name']] = df_property['LSOA_Code_Name'].str.split(' : ', n=1, expand=True)
df_property.drop(columns=['LSOA_Code_Name', 'LSOA_Name'], inplace=True)

# Convert all data columns to numeric
data_cols = [col for col in df_property.columns if col != 'LSOA_Code']
for col in data_cols:
    df_property[col] = pd.to_numeric(df_property[col].astype(str).str.replace(',', ''), errors='coerce')

df_property.dropna(inplace=True)


# --- 2. Engineer the 'Pct_Flats' Feature ---
# Sum all the different flat/apartment categories
flat_cols = ['Flats_Purpose_Built', 'Flats_Converted_Shared', 'Flats_Converted_Other', 'Flats_Commercial']
df_property['Total_Flats'] = df_property[flat_cols].sum(axis=1)

# Calculate the percentage
df_property['Pct_Flats'] = (df_property['Total_Flats'] / df_property['Total_Households']) * 100

In [12]:
df_property.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4988 entries, 0 to 4987
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Total_Households        4988 non-null   float64
 1   Flats_Purpose_Built     4988 non-null   float64
 2   Flats_Converted_Shared  4988 non-null   float64
 3   Flats_Converted_Other   4988 non-null   float64
 4   Flats_Commercial        4988 non-null   float64
 5   LSOA_Code               4988 non-null   object 
 6   Total_Flats             4988 non-null   float64
 7   Pct_Flats               4988 non-null   float64
dtypes: float64(7), object(1)
memory usage: 350.7+ KB


In [14]:
# Define the name for your new CSV file
output_path = 'df_property.csv'

# Save the DataFrame to a CSV file
# index=False prevents pandas from writing the DataFrame index as a column
df_property.to_csv(output_path, index=False)

print(f"DataFrame successfully saved to: {output_path}")

DataFrame successfully saved to: df_property.csv


In [18]:
import geopandas as gpd

# Define the path to your saved file
analysis_gdf_path = '/Users/callumhall/Documents/Dissertation/Datasets/analysis_ready_gdf.gpkg'

# Use geopandas.read_file() to load the GeoPackage
analysis_gdf = gpd.read_file(analysis_gdf_path)

# Verify it loaded correctly
print("GeoPackage file loaded successfully.")
analysis_gdf.info()

GeoPackage file loaded successfully.
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 319097 entries, 0 to 319096
Data columns (total 54 columns):
 #   Column                      Non-Null Count   Dtype   
---  ------                      --------------   -----   
 0   LSOA21CD                    319097 non-null  object  
 1   LSOA21NM                    319097 non-null  object  
 2   LSOA21NMW                   0 non-null       object  
 3   BNG_E                       319097 non-null  int64   
 4   BNG_N                       319097 non-null  int64   
 5   LAT                         319097 non-null  float64 
 6   LONG                        319097 non-null  float64 
 7   GlobalID                    319097 non-null  object  
 8   Total_Population            319097 non-null  int64   
 9   Edu_Level_4_Plus            319097 non-null  float64 
 10  LSOA_Code                   319097 non-null  object  
 11  Total_Households            319097 non-null  int64   
 12  Tenure_Privat

In [20]:
# --- 1. Define the paths to your files ---
analysis_gdf_path = '/Users/callumhall/Documents/Dissertation/Datasets/analysis_ready_gdf.gpkg'
property_data_path = '/Users/callumhall/Documents/Dissertation/Datasets/Census Data/df_property.csv' # Assumes this is in the same folder

# --- 2. Load your main GeoDataFrame and the new property data ---
print("Loading existing analysis_ready_gdf...")
analysis_gdf = gpd.read_file(analysis_gdf_path)

print("Loading cleaned property type data...")
df_property = pd.read_csv(property_data_path)

# --- 3. Merge the new feature into your main dataset ---
# The LSOA code in your analysis_gdf is likely named 'LSOA21CD' or 'LSOA_Code'.
# Let's check for the correct column name before merging.
lsoa_code_col = 'LSOA21CD' if 'LSOA21CD' in analysis_gdf.columns else 'LSOA_Code'

print(f"Merging using key: '{lsoa_code_col}'")
analysis_gdf = pd.merge(
    analysis_gdf,
    df_property[['LSOA_Code', 'Pct_Flats']],
    left_on=lsoa_code_col,
    right_on='LSOA_Code',
    how='left'
)

# --- 4. Verify and Save the updated file ---
print("\n--- 'Pct_Flats' feature added successfully ---")
# Check for the new column and some values
if 'Pct_Flats' in analysis_gdf.columns:
    print(analysis_gdf[['Borough', 'Pct_Flats']].head())
else:
    print("Warning: 'Pct_Flats' column was not added. Please check the merge keys.")


# Save the updated GeoDataFrame, overwriting the old file
analysis_gdf.to_file(analysis_gdf_path, driver='GPKG')
print(f"\nUpdated GeoDataFrame with property type data saved to: {analysis_gdf_path}")

Loading existing analysis_ready_gdf...
Loading cleaned property type data...
Merging using key: 'LSOA21CD'

--- 'Pct_Flats' feature added successfully ---
                Borough  Pct_Flats
0  Barking and Dagenham  26.714801
1  Barking and Dagenham  26.714801
2  Barking and Dagenham  26.714801
3  Barking and Dagenham  26.714801
4  Barking and Dagenham  26.714801

Updated GeoDataFrame with property type data saved to: /Users/callumhall/Documents/Dissertation/Datasets/analysis_ready_gdf.gpkg


In [22]:
import geopandas as gpd

# Define the path to your saved file
analysis_gdf_path = '/Users/callumhall/Documents/Dissertation/Datasets/analysis_ready_gdf.gpkg'

# Load the GeoDataFrame
analysis_gdf = gpd.read_file(analysis_gdf_path)

# --- The Verification Step ---

# Find the correct LSOA code column (it might have a suffix)
lsoa_code_column = None
for col in ['LSOA21CD', 'LSOA_Code', 'LSOA21CD_x']:
    if col in analysis_gdf.columns:
        lsoa_code_column = col
        break

if lsoa_code_column:
    # Count the total number of rows
    total_rows = len(analysis_gdf)
    
    # Count the number of unique LSOA codes
    unique_lsoas = analysis_gdf[lsoa_code_column].nunique()
    
    print(f"Total rows in the GeoDataFrame: {total_rows}")
    print(f"Number of unique LSOAs: {unique_lsoas}")
    
    if total_rows == unique_lsoas:
        print("\nConclusion: Your data is at the LSOA level. ✅")
    else:
        print("\nConclusion: Your data is NOT at the LSOA level, as there are duplicate rows.")
else:
    print("Error: Could not find a recognizable LSOA code column.")

Total rows in the GeoDataFrame: 319097
Number of unique LSOAs: 4988

Conclusion: Your data is NOT at the LSOA level, as there are duplicate rows.
